# Enterprise BFSI Client MDM Platform
# 02 - Contract Standardization

 Responsibilities:
   1. Read Bronze Delta tables
   2. Validate each source against its contract
   3. Separate invalid records
   4. Standardize valid records
   5. Build the canonical client representation
   6. Persist results for next step

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import DecimalType

CATALOG = "bfsi_mdm_dev"

SYSTEMS = [
    "core",
    "crm",
    "kyc",
    "wealth",
]

QUARANTINE_ROOT = (
    "abfss://quarantine@stbfsimdmdev.dfs.core.windows.net/"
)

print("Notebook 02 initialized.")
print(f"Catalog : {CATALOG}")
print(f"Sources : {SYSTEMS}")


Notebook 02 initialized.
Catalog : bfsi_mdm_dev
Sources : ['core', 'crm', 'kyc', 'wealth']


####Verify the four Bronze tables created

In [0]:
spark.sql(
    f"SHOW TABLES IN {CATALOG}.bronze"
).show(truncate=False)

+--------+---------+-----------+
|database|tableName|isTemporary|
+--------+---------+-----------+
|bronze  |core     |false      |
|bronze  |crm      |false      |
|bronze  |kyc      |false      |
|bronze  |wealth   |false      |
+--------+---------+-----------+



In [0]:
for system in SYSTEMS:

    df = spark.table(
        f"{CATALOG}.bronze.{system}"
    )

    print(
        f"{system.upper():<8} "
        f"{df.count():,} rows"
    )
    

CORE     964,240 rows
CRM      660,223 rows
KYC      811,892 rows
WEALTH   202,544 rows


####Source contracts

These describe the SOURCE representation.

They are not the canonical Silver schema.

In [0]:
DATE_FORMATS = [
    "yyyy-MM-dd",
    "dd/MM/yyyy",
    "dd-MM-yyyy",
]

CONTRACTS = {

    "core": {

        "customer_id": {
            "required": True,
            "nullable": False,
            "pattern": r"^C[0-9]{8}$",
        },

        "full_name": {
            "required": True,
            "nullable": False,
        },

        "dob": {
            "required": True,
            "nullable": True,
            "date_formats": DATE_FORMATS,
        },

        "phone": {
            "required": True,
            "nullable": False,
        },

        "email": {
            "required": True,
            "nullable": True,
        },

        "address": {
            "required": True,
            "nullable": False,
        },

        "account_id": {
            "required": True,
            "nullable": False,
        },

        "branch_id": {
            "required": True,
            "nullable": False,
        },
    },

    "crm": {

        "crm_customer_id": {
            "required": True,
            "nullable": False,
        },

        "customer_name": {
            "required": True,
            "nullable": False,
        },

        "date_of_birth": {
            "required": True,
            "nullable": True,
            "date_formats": DATE_FORMATS,
        },

        "mobile": {
            "required": True,
            "nullable": False,
        },

        "email_address": {
            "required": True,
            "nullable": True,
        },

        "residential_address": {
            "required": True,
            "nullable": False,
        },

        "segment": {
            "required": True,
            "nullable": False,
        },
    },

    "kyc": {

        "kyc_id": {
            "required": True,
            "nullable": False,
        },

        "legal_name": {
            "required": True,
            "nullable": False,
        },

        "dob": {
            "required": True,
            "nullable": True,
            "date_formats": DATE_FORMATS,
        },

        "tax_id": {
            "required": True,
            "nullable": False,
        },

        "phone": {
            "required": True,
            "nullable": False,
        },

        "address": {
            "required": True,
            "nullable": False,
        },

        "kyc_status": {
            "required": True,
            "nullable": False,
        },

        "verification_date": {
            "required": True,
            "nullable": False,
            "date_formats": DATE_FORMATS,
        },

        "LEI": {
            "required": True,
            "nullable": True,
            "pattern": r"^[A-Z0-9]{20}$",
        },
    },

    "wealth": {

        "client_id": {
            "required": True,
            "nullable": False,
        },

        "client_name": {
            "required": True,
            "nullable": False,
        },

        "dob": {
            "required": True,
            "nullable": True,
            "date_formats": DATE_FORMATS,
        },

        "phone": {
            "required": True,
            "nullable": False,
        },

        "email": {
            "required": True,
            "nullable": True,
        },

        "address": {
            "required": True,
            "nullable": False,
        },

        "aum": {
            "required": True,
            "nullable": False,
            "type": "decimal",
        },

        "risk_profile": {
            "required": True,
            "nullable": False,
        },

        "relationship_manager": {
            "required": True,
            "nullable": False,
        },
    },
}

####Generic contract validator

In [0]:
def validate_contract(df, contract):
    """
    Apply source-contract validation using native Spark expressions.

    Every rule produces either:
        NULL
        or
        an explanatory error message.

    All errors are collected into:
        contract_issues
    """

    issue_expressions = []

    for field_name, rules in contract.items():

        col = F.col(field_name)

        # NULL OR whitespace-only values are considered blank.
        is_blank = (
            col.isNull()
            | (F.trim(col) == "")
        )

        # --------------------------------------------------------
        # Required + non-nullable
        # --------------------------------------------------------
        if (
            rules.get("required", False)
            and not rules.get("nullable", False)
        ):

            issue_expressions.append(
                F.when(
                    is_blank,
                    F.lit(
                        f"{field_name}: required value is blank"
                    )
                )
            )

        # --------------------------------------------------------
        # Pattern validation
        # --------------------------------------------------------
        if "pattern" in rules:

            issue_expressions.append(
                F.when(
                    (~is_blank)
                    &
                    (~col.rlike(
                        rules["pattern"]
                    )),
                    F.lit(
                        f"{field_name}: invalid pattern"
                    )
                )
            )

        # --------------------------------------------------------
        # Date semantic validation
        #
        # A value passes if at least ONE accepted format parses.
        # --------------------------------------------------------
        
        if "date_formats" in rules:

            parsed_date = F.coalesce(
                *[
                    F.try_to_date(
                        col,
                        F.lit(fmt)
                    )
                    for fmt in rules["date_formats"]
                ]
            )

            issue_expressions.append(
                F.when(
                    (~is_blank)
                    &
                    parsed_date.isNull(),
                    F.lit(
                        f"{field_name}: invalid date format"
                    )
                )
            )


        

        # --------------------------------------------------------
        # Numeric validation
        #
        # Wealth AUM must be parseable as a numeric value.
        # --------------------------------------------------------
        if rules.get("type") == "decimal":

            parsed_decimal = F.expr(
                f"try_cast(`{field_name}` AS DECIMAL(18,2))"
            )

            issue_expressions.append(
                F.when(
                    (~is_blank)
                    &
                    parsed_decimal.isNull(),
                    F.lit(
                        f"{field_name}: invalid numeric value"
                    )
                )
            )
        

    # Convert all individual rule outputs into one array.
    #
    # Example:
    #
    # [NULL, "dob: invalid date format", NULL]
    #
    # array_compact()
    #
    # becomes:
    #
    # ["dob: invalid date format"]
    #
    return df.withColumn(
        "contract_issues",
        F.array_compact(
            F.array(*issue_expressions)
        )
    )

####Create source-to-canonical mapping

In [0]:
CANONICAL_MAPPING = {

    "core": {
        "id": "customer_id",
        "name": "full_name",
        "dob": "dob",
        "phone": "phone",
        "email": "email",
        "address": "address",
        "tax_id": None,
        "lei": None,
        "account_id": "account_id",
        "branch_id": "branch_id",
        "segment": None,
        "kyc_status": None,
        "verification_date": None,
        "aum": None,
        "risk_profile": None,
        "relationship_manager": None,
    },

    "crm": {
        "id": "crm_customer_id",
        "name": "customer_name",
        "dob": "date_of_birth",
        "phone": "mobile",
        "email": "email_address",
        "address": "residential_address",
        "tax_id": None,
        "lei": None,
        "account_id": None,
        "branch_id": None,
        "segment": "segment",
        "kyc_status": None,
        "verification_date": None,
        "aum": None,
        "risk_profile": None,
        "relationship_manager": None,
    },

    "kyc": {
        "id": "kyc_id",
        "name": "legal_name",
        "dob": "dob",
        "phone": "phone",
        "email": None,
        "address": "address",
        "tax_id": "tax_id",
        "lei": "LEI",
        "account_id": None,
        "branch_id": None,
        "segment": None,
        "kyc_status": "kyc_status",
        "verification_date": "verification_date",
        "aum": None,
        "risk_profile": None,
        "relationship_manager": None,
    },

    "wealth": {
        "id": "client_id",
        "name": "client_name",
        "dob": "dob",
        "phone": "phone",
        "email": "email",
        "address": "address",
        "tax_id": None,
        "lei": None,
        "account_id": None,
        "branch_id": None,
        "segment": None,
        "kyc_status": None,
        "verification_date": None,
        "aum": "aum",
        "risk_profile": "risk_profile",
        "relationship_manager": "relationship_manager",
    },
}

####Build the normalization functions

In [0]:
def normalize_name_col(col):
    """
    Name normalization:
      1. Collapse repeated whitespace
      2. Remove leading/trailing spaces
      3. Standardize casing
    """

    collapsed = F.trim(
        F.regexp_replace(
            col,
            r"\s+",
            " "
        )
    )

    return F.initcap(collapsed)


def normalize_phone_col(col):
    """
    Remove formatting characters from phone numbers.

    Example:
        +44 7123-456789
             ↓
        447123456789
             ↓
        keep final 10 digits
    """

    digits_only = F.regexp_replace(
        col,
        r"\D",
        ""
    )

    return F.when(
        F.length(digits_only) > 10,
        F.substring(
            digits_only,
            -10,
            10
        )
    ).otherwise(
        digits_only
    )


def normalize_email_col(col):
    """
    Email normalization:
      trim spaces
      lowercase
    """

    return F.lower(
        F.trim(col)
    )


def normalize_address_col(col):
    """
    Address normalization:
      1. collapse whitespace
      2. normalize comma spacing
    """

    collapsed = F.trim(
        F.regexp_replace(
            col,
            r"\s+",
            " "
        )
    )

    return F.regexp_replace(
        collapsed,
        r"\s*,\s*",
        ", "
    )


def normalize_date_col(col):
    """
    Convert all accepted source date formats
    to one canonical string:

        yyyy-MM-dd

    Invalid values return NULL instead of
    raising an ANSI parsing exception.
    """

    parsed = F.coalesce(
        *[
            F.try_to_date(
                col,
                F.lit(fmt)
            )
            for fmt in DATE_FORMATS
        ]
    )

    return F.date_format(
        parsed,
        "yyyy-MM-dd"
    )

####Build the canonical DataFrame

In [0]:
def source_column(mapping, field_name):
    """
    Return the source column for a canonical attribute.

    If the source does not provide that attribute,
    return a typed NULL.

    NULL is used here because downstream functions such as
    date parsing and decimal casting can safely operate on NULL.
    The final canonical DataFrame converts NULL string fields
    to empty strings with fillna("").
    """

    source_name = mapping.get(field_name)

    if source_name is None:
        return F.lit(None).cast("string")

    return F.col(source_name)


def build_canonical_dataframe(
    df,
    system
):
    """
    Convert one source's validated records into
    the common canonical representation.
    """

    mapping = CANONICAL_MAPPING[system]

    return (
        df.select(

            # --------------------------------------------
            # Source identity
            # --------------------------------------------
            F.lit(
                system.upper()
            ).alias(
                "source_system"
            ),

            source_column(
                mapping,
                "id"
            ).alias(
                "source_id"
            ),

            # --------------------------------------------
            # Identity attributes
            # --------------------------------------------
            source_column(
                mapping,
                "name"
            ).alias(
                "name"
            ),

            normalize_name_col(
                source_column(
                    mapping,
                    "name"
                )
            ).alias(
                "normalized_name"
            ),

            source_column(
                mapping,
                "dob"
            ).alias(
                "dob"
            ),

            normalize_date_col(
                source_column(
                    mapping,
                    "dob"
                )
            ).alias(
                "normalized_dob"
            ),

            source_column(
                mapping,
                "phone"
            ).alias(
                "phone"
            ),

            normalize_phone_col(
                source_column(
                    mapping,
                    "phone"
                )
            ).alias(
                "normalized_phone"
            ),

            source_column(
                mapping,
                "email"
            ).alias(
                "email"
            ),

            normalize_email_col(
                source_column(
                    mapping,
                    "email"
                )
            ).alias(
                "normalized_email"
            ),

            source_column(
                mapping,
                "address"
            ).alias(
                "address"
            ),

            normalize_address_col(
                source_column(
                    mapping,
                    "address"
                )
            ).alias(
                "normalized_address"
            ),

            # --------------------------------------------
            # Other business attributes
            # --------------------------------------------
            source_column(
                mapping,
                "tax_id"
            ).alias(
                "tax_id"
            ),

            source_column(
                mapping,
                "lei"
            ).alias(
                "lei"
            ),

            source_column(
                mapping,
                "account_id"
            ).alias(
                "account_id"
            ),

            source_column(
                mapping,
                "branch_id"
            ).alias(
                "branch_id"
            ),

            source_column(
                mapping,
                "segment"
            ).alias(
                "segment"
            ),

            source_column(
                mapping,
                "kyc_status"
            ).alias(
                "kyc_status"
            ),

            source_column(
                mapping,
                "verification_date"
            ).alias(
                "verification_date"
            ),

            normalize_date_col(
                source_column(
                    mapping,
                    "verification_date"
                )
            ).alias(
                "normalized_verification_date"
            ),

            (
                F.expr(
                    f"try_cast(`{mapping['aum']}` AS DECIMAL(18,2))"
                )
                if mapping.get("aum") is not None
                else F.lit(None).cast("decimal(18,2)")
            ).alias(
                "aum"
            ),           
            

            source_column(
                mapping,
                "risk_profile"
            ).alias(
                "risk_profile"
            ),

            source_column(
                mapping,
                "relationship_manager"
            ).alias(
                "relationship_manager"
            ),
        )

        # Convert actual NULLs to empty strings for canonical
        # comparison fields.
        .fillna("")
    )

####Contract-valid records to quarantine and staging

In [0]:
def process_source(system):

    print("=" * 80)
    print(f"PROCESSING SOURCE: {system.upper()}")
    print("=" * 80)

    # --------------------------------------------
    # Read Bronze
    # --------------------------------------------
    bronze_df = spark.table(
        f"{CATALOG}.bronze.{system}"
    )

    input_count = bronze_df.count()

    print(
        f"Bronze input rows: {input_count:,}"
    )

    # --------------------------------------------
    # Contract validation
    # --------------------------------------------
    checked_df = validate_contract(
        bronze_df,
        CONTRACTS[system]
    )

    # --------------------------------------------
    # PASS records
    # --------------------------------------------
    passed_df = (
        checked_df
        .filter(
            F.size("contract_issues") == 0
        )
        .drop("contract_issues")
    )

    # --------------------------------------------
    # FAIL records
    # --------------------------------------------
    failed_df = (
        checked_df
        .filter(
            F.size("contract_issues") > 0
        )
        .withColumn(
            "failure_reason",
            F.concat_ws(
                "; ",
                "contract_issues"
            )
        )
    )

    pass_count = passed_df.count()
    fail_count = failed_df.count()

    print(
        f"Contract PASS : {pass_count:,}"
    )
    print(
        f"Contract FAIL : {fail_count:,}"
    )

    # ------------------------------------------------
    # Reconciliation:
    #
    # input = passed + failed
    # ------------------------------------------------
    assert (
        input_count
        ==
        pass_count + fail_count
    ), (
        f"Contract reconciliation failed "
        f"for {system}"
    )

    # --------------------------------------------
    # Quarantine invalid records
    # --------------------------------------------
    (
        failed_df
        .write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(
            f"{CATALOG}.quarantine.contract_{system}"
        )
    )

    # --------------------------------------------
    # Standardize valid records
    # --------------------------------------------
    canonical_df = build_canonical_dataframe(
        passed_df,
        system
    )

    # --------------------------------------------
    # Persist staging result
    # --------------------------------------------
    (
        canonical_df
        .write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(
            f"{CATALOG}.silver.stg_{system}_standardized"
        )
    )

    print(
        f"Staging table written: "
        f"{CATALOG}.silver.stg_{system}_standardized"
    )

    print("=" * 80)

    return canonical_df

####Run the entire Silver standardization process

In [0]:
canonical_outputs = {}

for system in SYSTEMS:

    canonical_outputs[system] = process_source(
        system
    )

PROCESSING SOURCE: CORE
Bronze input rows: 964,240
Contract PASS : 964,240
Contract FAIL : 0
Staging table written: bfsi_mdm_dev.silver.stg_core_standardized
PROCESSING SOURCE: CRM
Bronze input rows: 660,223
Contract PASS : 660,223
Contract FAIL : 0
Staging table written: bfsi_mdm_dev.silver.stg_crm_standardized
PROCESSING SOURCE: KYC
Bronze input rows: 811,892
Contract PASS : 811,890
Contract FAIL : 2
Staging table written: bfsi_mdm_dev.silver.stg_kyc_standardized
PROCESSING SOURCE: WEALTH
Bronze input rows: 202,544
Contract PASS : 202,541
Contract FAIL : 3
Staging table written: bfsi_mdm_dev.silver.stg_wealth_standardized


####Verify the four staging tables

In [0]:
spark.sql(
    f"SHOW TABLES IN {CATALOG}.silver"
).show(
    truncate=False
)

+--------+-----------------------+-----------+
|database|tableName              |isTemporary|
+--------+-----------------------+-----------+
|silver  |stg_core_standardized  |false      |
|silver  |stg_crm_standardized   |false      |
|silver  |stg_kyc_standardized   |false      |
|silver  |stg_wealth_standardized|false      |
+--------+-----------------------+-----------+



In [0]:
for system in SYSTEMS:

    df = spark.table(
        f"{CATALOG}.silver.stg_{system}_standardized"
    )

    print(
        f"{system.upper():<8} "
        f"{df.count():,}"
    )

CORE     964,240
CRM      660,223
KYC      811,890
WEALTH   202,541


####Verify canonical representation

In [0]:
display(
    spark.table(
        f"{CATALOG}.silver.stg_core_standardized"
    ).select(
        "source_system",
        "source_id",
        "name",
        "normalized_name",
        "dob",
        "normalized_dob",
        "phone",
        "normalized_phone",
        "email",
        "normalized_email",
        "address",
        "normalized_address"
    ).limit(10)
)

source_system,source_id,name,normalized_name,dob,normalized_dob,phone,normalized_phone,email,normalized_email,address,normalized_address
CORE,C00000001,Connor Smith,Connor Smith,1971-10-19,1971-10-19,+44(0)20 7496 0962,2074960962,elaine13@example.org,elaine13@example.org,"Studio 8 Hopkins park, West Scott, South East, RG16 2RP","Studio 8 Hopkins park, West Scott, South East, RG16 2RP"
CORE,C00000002,Kevin Ruth Hayes,Kevin Ruth Hayes,1970-05-26,1970-05-26,01632 960381,1632960381,jasminehopkins@example.net,jasminehopkins@example.net,"881 Davies hollow, Charlesborough, South East, SO6 2NK","881 Davies hollow, Charlesborough, South East, SO6 2NK"
CORE,C00000003,Jemma Dean,Jemma Dean,1957-03-03,1957-03-03,+44117 4960356,1174960356,lewisbrian@example.net,lewisbrian@example.net,"7 Donald trace, Wallacefurt, Scotland, PA5 2ZM","7 Donald trace, Wallacefurt, Scotland, PA5 2ZM"
CORE,C00000004,Richard Duncan Chambers,Richard Duncan Chambers,1983-10-21,1983-10-21,(0909) 879 0389,9098790389,walkergraham@example.org,walkergraham@example.org,"187 Fowler via, North Shaneborough, South East, TN17 7LC","187 Fowler via, North Shaneborough, South East, TN17 7LC"
CORE,C00000005,Harry Sharpe,Harry Sharpe,1967-09-22,1967-09-22,+44161 496 0148,1614960148,jenna56@example.org,jenna56@example.org,"Studio 5 Olivia motorway, Geraldineberg, East Midlands, NG8 6JP","Studio 5 Olivia motorway, Geraldineberg, East Midlands, NG8 6JP"
CORE,C00000006,Keith Francis Wilson,Keith Francis Wilson,1984-05-13,1984-05-13,(0141) 4960037,1414960037,ndavey@example.org,ndavey@example.org,"0 Welch spurs, Stewarttown, London, W12 6VP","0 Welch spurs, Stewarttown, London, W12 6VP"
CORE,C00000007,Mitchell Janice Blackburn,Mitchell Janice Blackburn,1974-06-16,1974-06-16,(0113)4960059,1134960059,darrendouglas@example.com,darrendouglas@example.com,"Studio 9 Simmons rapids, Port Ashley, Scotland, DD13 2GW","Studio 9 Simmons rapids, Port Ashley, Scotland, DD13 2GW"
CORE,C00000008,Raymond Marshall,Raymond Marshall,1988-06-26,1988-06-26,+4429 2018 0580,2920180580,dayraymond@example.net,dayraymond@example.net,"714 Yvonne manor, East Justin, North West, SK2 7HW","714 Yvonne manor, East Justin, North West, SK2 7HW"
CORE,C00000009,Leanne Robert Stevens,Leanne Robert Stevens,1982-04-11,1982-04-11,+44(0)9098790874,9098790874,robertscarol@example.com,robertscarol@example.com,"Studio 88J Sara wells, East Jade, London, WC16 8FG","Studio 88J Sara wells, East Jade, London, WC16 8FG"
CORE,C00000010,Dean Reynolds,Dean Reynolds,1978-02-07,1978-02-07,+44(0)1914960387,1914960387,damianrowe@example.net,damianrowe@example.net,"Flat 75H Ali union, Georgiachester, North West, CH20 7ZU","Flat 75H Ali union, Georgiachester, North West, CH20 7ZU"
